In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
import torch
import os

import gym
from gym import spaces
from PortfolioEnv2 import PortfolioEnv2
from stable_baselines3 import PPO, SAC
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.monitor import Monitor


In [ ]:
#Seeding
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [ ]:
data = pd.read_csv('all_sector_data-4-15.csv', parse_dates=['Date'], index_col='Date')
data.sort_index(inplace=True)

asset_cols = ['XLC', 'XLY', 'XLP', 'XLE', 'XLF', 'XLV', 'XLI', 'XLK', 'XLB', 'XLRE', 'XLU']
vol_cols = ['vol20', 'vol60', 'VIX']

In [ ]:
#Split into train/test
train_start = "2018-09-13"
train_end   = "2024-01-02"
test_start  = "2024-01-03"
test_end    = "2025-04-15"
train_data = data.loc[train_start:train_end]
test_data  = data.loc[test_start:test_end]

In [ ]:
from stable_baselines3.common.callbacks import BaseCallback

class CumulativeSharpeCallback(BaseCallback):
    def __init__(self, verbose=0):
        super(CumulativeSharpeCallback, self).__init__(verbose)
        self.cumulative_reward = 0.0

    def _on_step(self) -> bool:
        rewards = self.locals.get('rewards', [])
        if rewards is not None:
            self.cumulative_reward += sum(rewards)

        self.logger.record('train/cumulative_reward', self.cumulative_reward)

        return True

In [ ]:
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize, VecMonitor

for time_steps in [100_000, 500_000, 1_000_000]: # Can remove -> ended using 1,000,000
    make_env = lambda: PortfolioEnv2(train_data, asset_cols, vol_cols, lookback=60)

    train_env = DummyVecEnv([make_env])
    train_env = VecMonitor(train_env)         
    vec_env = VecNormalize(train_env, norm_obs=True, norm_reward=True, clip_reward=1.0)

    vec_env.save(f"ppo_vec_normalize_{time_steps}.pkl")

    model = PPO(
        "MlpPolicy",
        vec_env,
        learning_rate=0.00026903861770121747,
        n_steps=512,
        batch_size=64,
        gamma=0.99,
        gae_lambda=0.924622179181684,
        clip_range=0.19524213935780285,
        ent_coef=0.003119711761086009,
        verbose=1,
        seed=SEED,
        tensorboard_log="./tb_logs/final_ppo_run",
    )

    model.learn(total_timesteps=time_steps, callback=CumulativeSharpeCallback(), log_interval=10)
    
    model.save(f"ppo_model{time_steps}.zip")


In [ ]:
from stable_baselines3 import SAC
from stable_baselines3.common.vec_env import DummyVecEnv, VecMonitor, VecNormalize

for time_steps in [100_000, 500_000, 1_000_000]:

    make_env = lambda: PortfolioEnv2(
        train_data, asset_cols, vol_cols, lookback=60
    )
    train_env = DummyVecEnv([make_env])       
    train_env = VecMonitor(train_env)         
    vec_env   = VecNormalize(
        train_env,
        norm_obs=True,
        norm_reward=True,
        clip_reward=1.0,
    )
    vec_env.save(f"vecnorm_sac_{time_steps}.pkl")

    sac_model = SAC(
        "MlpPolicy",
        vec_env,
        verbose=1,
        learning_rate=0.000197,
        tau=0.00503,
        buffer_size=200_000,
        batch_size=64,
        train_freq=1,
        ent_coef=0.0571,
        gamma=0.9110,
        gradient_steps=1,
        seed=SEED,
        tensorboard_log="./tb_logs/final_sac_run",
    )

    sac_model.learn(
        total_timesteps=time_steps,
        callback=CumulativeSharpeCallback(),
        log_interval=10
    )
    
    sac_model.save(f"sac_model_{time_steps}.zip")